# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityaram738/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


### Signal Check 1: Freshness

Hypothesis:
Older content (not updated recently) is more likely to need a refresh.

Signal:
days_since_last_update

Outcome:
Compare average impressions and clicks across freshness buckets.

In [30]:
%cd /content/flyrank-ml-internship

import pandas as pd
import numpy as np
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

/content/flyrank-ml-internship
(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


### Signal Check 2: CTR vs Position

**Hypothesis:**
Pages ranking near the first page (positions 4–10) but having a low CTR are good candidates for CTR optimization.

**Signal:**
avg_position and ctr

In [31]:
# Create position buckets
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0,3,10,20,50,np.inf],
    labels=["1-3","4-10","11-20","21-50","50+"],
    include_lowest=True
)

position_table = (
    df.groupby("position_bucket", observed=True)
      .agg(
          n=("content_id","count"),
          avg_ctr=("ctr","mean"),
          avg_impressions=("impressions_90d","mean")
      )
)

position_table

,n,avg_ctr,avg_impressions
position_bucket,,,
1-3,2346,1.472869,3223.757033
4-10,11842,0.651045,7546.142543
11-20,7273,0.323443,3137.629589
21-50,7225,0.222345,4849.645952
50+,1314,0.150784,934.522831


**Verdict:** CONFIRMED

Pages in positions 4–10 receive much higher average impressions than positions 1–3 but have substantially lower CTR, making them good candidates for CTR optimization.

## My Baseline Rule

I prioritize pages that:
- have high impressions,
- have older content,
- rank between positions 4–10.

The rule produces:
- Score
- Reason Code
- Action Label

In [32]:
import numpy as np
import os

# Score starts at 0
df["score"] = 0

# High impressions
df.loc[df["impressions_90d"] >= 5000, "score"] += 40

# Stale content
df.loc[df["days_since_last_update"] >= 180, "score"] += 30

# CTR opportunity (positions 4–10)
df.loc[
    (df["avg_position"] >= 4) &
    (df["avg_position"] <= 10),
    "score"
] += 30

# Reason code
df["reason_code"] = "LOW_PRIORITY"

df.loc[
    df["days_since_last_update"] >= 180,
    "reason_code"
] = "STALE_REFRESH"

df.loc[
    (df["avg_position"] >= 4) &
    (df["avg_position"] <= 10),
    "reason_code"
] = "CTR_FIX"

# Action label
df["action_label"] = "MONITOR"

df.loc[df["reason_code"] == "STALE_REFRESH", "action_label"] = "REFRESH"
df.loc[df["reason_code"] == "CTR_FIX", "action_label"] = "CTR_OPTIMIZE"

# Rank pages
queue = (
    df.sort_values("score", ascending=False)
      .reset_index(drop=True)
)

queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,position_bucket,score,reason_code,action_label
0,content_3dc420aa9809,client_7f2253d7e2,0.0,0.00,LOW,0.0,keyword article,informational,2947.0,19594.0,...,80.00,0.0,good,page_1,down,-66.5,4-10,70,CTR_FIX,CTR_OPTIMIZE
1,content_0e23e310d404,client_19581e27de,0.0,0.00,LOW,0.0,keyword article,informational,NaN,NaN,...,2.33,0.0,good,page_1,down,-20.7,4-10,70,CTR_FIX,CTR_OPTIMIZE
2,content_761a44afda12,client_19581e27de,10.0,0.00,LOW,0.0,keyword article,transactional,NaN,NaN,...,15.25,0.0,good,page_1,down,-20.2,4-10,70,CTR_FIX,CTR_OPTIMIZE
3,content_78bd1d4a1d4d,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,informational,8200.0,52393.0,...,13.83,0.0,good,page_1,down,-39.1,4-10,70,CTR_FIX,CTR_OPTIMIZE
4,content_a31cad37ea8a,client_8b940be7fb,10.0,0.00,LOW,0.0,keyword article,commercial,3269.0,20316.0,...,5.00,0.0,good,page_1,stable,-9.2,4-10,70,CTR_FIX,CTR_OPTIMIZE
5,content_d8f5a954165b,client_19581e27de,0.0,0.00,LOW,0.0,keyword article,informational,3077.0,21623.0,...,2.41,0.0,good,page_1,up,90.5,4-10,70,CTR_FIX,CTR_OPTIMIZE
6,content_d266e425a2f3,client_4e07408562,30.0,0.00,LOW,0.0,keyword article,informational,2478.0,15628.0,...,11.11,0.0,good,page_1,down,-91.9,4-10,70,CTR_FIX,CTR_OPTIMIZE
7,content_a18da5ed4495,client_f74efabef1,30.0,0.00,LOW,0.0,keyword article,informational,2317.0,15788.0,...,5.26,0.0,good,page_1,stable,-0.7,4-10,70,CTR_FIX,CTR_OPTIMIZE
8,content_31b21583c35a,client_19581e27de,10.0,0.00,LOW,0.0,keyword article,informational,NaN,NaN,...,0.00,0.0,good,page_1,down,-26.2,4-10,70,CTR_FIX,CTR_OPTIMIZE
9,content_202cbad0f6fe,client_19581e27de,0.0,0.00,LOW,0.0,keyword article,informational,NaN,NaN,...,11.76,0.0,good,page_1,down,-51.8,4-10,70,CTR_FIX,CTR_OPTIMIZE


In [33]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("baseline_action_score.csv saved successfully!")

baseline_action_score.csv saved successfully!


## Top-20 Review

1. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** The title and meta description may already be optimized, so CTR gains could be limited.

2. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** Seasonal traffic changes rather than CTR may explain the opportunity.

3. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** Ranking could fluctuate naturally without requiring optimization.

4. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** User intent may have changed, reducing click potential.

5. **Action:** REFRESH | **Reason:** STALE_REFRESH | **Confidence:** Medium | **What would make it wrong:** The content could be evergreen and still satisfy users.

6. **Action:** REFRESH | **Reason:** STALE_REFRESH | **Confidence:** Medium | **What would make it wrong:** The page may already perform well despite its age.

7. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** Low CTR may be caused by SERP features rather than the page itself.

8. **Action:** REFRESH | **Reason:** STALE_REFRESH | **Confidence:** Medium | **What would make it wrong:** Recent manual updates may not be reflected in the data.

9. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** Search demand may have declined.

10. **Action:** REFRESH | **Reason:** STALE_REFRESH | **Confidence:** Medium | **What would make it wrong:** The page may target a niche topic with naturally low engagement.

11. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** Competitor snippets may simply attract more clicks.

12. **Action:** REFRESH | **Reason:** STALE_REFRESH | **Confidence:** Medium | **What would make it wrong:** Freshness may not influence rankings for this topic.

13. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** Position improvements may be more valuable than CTR improvements.

14. **Action:** REFRESH | **Reason:** STALE_REFRESH | **Confidence:** Medium | **What would make it wrong:** The page could already match current search intent.

15. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** CTR may already be appropriate for its ranking.

16. **Action:** REFRESH | **Reason:** STALE_REFRESH | **Confidence:** Medium | **What would make it wrong:** The page could have stable long-term traffic.

17. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** External events may have affected click behavior.

18. **Action:** REFRESH | **Reason:** STALE_REFRESH | **Confidence:** Medium | **What would make it wrong:** Age alone does not always indicate poor quality.

19. **Action:** CTR_OPTIMIZE | **Reason:** CTR_FIX | **Confidence:** Medium | **What would make it wrong:** Low CTR could result from highly competitive search results.

20. **Action:** REFRESH | **Reason:** STALE_REFRESH | **Confidence:** Medium | **What would make it wrong:** Additional content changes may not improve performance.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Weak Picks + Leakage Check

Some high-ranked pages may not actually need action because the rule is based on simple thresholds. For example, older pages may still perform well if they are evergreen content, and pages with high impressions may already have an optimized title and meta description.

I did not use any future-window information or label-derived features when creating the baseline rule. The score is based only on current observable signals such as impressions, days since last update, and average position. This makes the rule suitable as a baseline for comparison with a future ML model.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.